### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from pathlib import Path

c:\Users\hhh\Desktop\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: attention.pdf


  ✓ Loaded 1 pages

Processing: embeddings.pdf
  ✓ Loaded 1 pages

Processing: objectdetection.pdf
  ✓ Loaded 1 pages

Processing: proposal.pdf
  ✓ Loaded 2 pages

Total documents loaded: 5


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T04:13:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T04:13:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAbstract\nA novel neural network architecture based solely on attention mechanisms, dispensing with\nrecurrence and convolutions entirely. The Transformer model achieves superior results on\nmachine translation tasks.\nKey Concept: Self-Attention\n\x7f Query (Q), Key (K), Value (V) matrices computed from input\n\x7f Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V\n\x7f Allows each word to attend to every other word in the sequence\nArchitecture\n\x7f Encoder: 6 identical layers, each with multi-head attention +

In [6]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)

Split 5 documents into 6 chunks

Example chunk:
Content: Attention Is All You Need
Abstract
A novel neural network architecture based solely on attention mechanisms, dispensing with
recurrence and convolutions entirely. The Transformer model achieves superi...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T04:13:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T04:13:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}


In [8]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T04:13:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T04:13:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAbstract\nA novel neural network architecture based solely on attention mechanisms, dispensing with\nrecurrence and convolutions entirely. The Transformer model achieves superior results on\nmachine translation tasks.\nKey Concept: Self-Attention\n\x7f Query (Q), Key (K), Value (V) matrices computed from input\n\x7f Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V\n\x7f Allows each word to attend to every other word in the sequence\nArchitecture\n\x7f Encoder: 6 identical layers, each with multi-head attention +

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 823.77it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\hhh\AppData\Local\Temp\ipykernel_6076\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 24


In [12]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-15T04:13:32+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-15T04:13:32+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\attention.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'attention.pdf', 'file_type': 'pdf'}, page_content='Attention Is All You Need\nAbstract\nA novel neural network architecture based solely on attention mechanisms, dispensing with\nrecurrence and convolutions entirely. The Transformer model achieves superior results on\nmachine translation tasks.\nKey Concept: Self-Attention\n\x7f Query (Q), Key (K), Value (V) matrices computed from input\n\x7f Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V\n\x7f Allows each word to attend to every other word in the sequence\nArchitecture\n\x7f Encoder: 6 identical layers, each with multi-head attention +

In [13]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 6 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:02<00:00,  2.06s/it]

Generated embeddings with shape: (6, 384)
Adding 6 documents to vector store...
Successfully added 6 documents to vector store
Total documents in collection: 30


In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [15]:
rag_retriever

In [16]:
rag_retriever.retrieve("What is attention is all you need")

Retrieving documents for query: 'What is attention is all you need'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.34it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [17]:
class RAGRetriever:
    def __init__(self, vectorstore, embedding_manager):
        self.vectorstore = vectorstore
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 3, score_threshold: float = 0.0):
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])
        
        # Direct chromadb query — no score filtering issue
        results = self.vectorstore.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )
        
        docs = results['documents'][0] if results['documents'] else []
        ids = results['ids'][0] if results['ids'] else []
        distances = results['distances'][0] if results['distances'] else []
        
        if not docs:
            print("Retrieved 0 documents")
            return []
        
        # Convert distance to similarity score
        retrieved = []
        for doc, id, dist in zip(docs, ids, distances):
            similarity = 1 / (1 + dist)  # Convert distance to similarity
            retrieved.append({
                'content': doc,
                'metadata': {'source': id},
                'similarity_score': similarity
            })
        
        print(f"Retrieved {len(retrieved)} documents ✅")
        return retrieved

# Recreate retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

# Test
results = rag_retriever.retrieve("Unified Multi-task Learning Framework", top_k=3)
for r in results:
    print(r['content'][:150])
    print(f"Score: {r['similarity_score']:.3f}")
    print("---")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 3
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 32.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents ✅
 BLEU score: 41.0 on WMT 2014 English-to-French
 Training time: 12 hours on 8 P100 GPUs
Key Takeaway
Attention mechanisms alone are sufficient for s
Score: 0.425
---
 BLEU score: 41.0 on WMT 2014 English-to-French
 Training time: 12 hours on 8 P100 GPUs
Key Takeaway
Attention mechanisms alone are sufficient for s
Score: 0.425
---
 BLEU score: 41.0 on WMT 2014 English-to-French
 Training time: 12 hours on 8 P100 GPUs
Key Takeaway
Attention mechanisms alone are sufficient for s
Score: 0.425
---


In [18]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 32.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents ✅


[{'content': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechanisms alone are sufficient for sequence modelling. No RNN or CNN needed.\nEnables full parallelisation during training.',
  'metadata': {'source': 'doc_67473df9_1'},
  'similarity_score': 0.4249186492740233},
 {'content': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechanisms alone are sufficient for sequence modelling. No RNN or CNN needed.\nEnables full parallelisation during training.',
  'metadata': {'source': 'doc_8bd46f7c_1'},
  'similarity_score': 0.4249186492740233},
 {'content': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechanisms alone are sufficient for sequence modelling. No RNN or CNN needed.\nEnables full parallelisation during training.',
  'metadata': {'source': 'doc_6b08346

Ingestion vectordb Context Pipeline with LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [ ]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)
print("LLM ready! ✅")

LLM ready! ✅


In [ ]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 40.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents ✅


The attention mechanism is a key concept in the Transformer model, which allows each word in a sequence to attend to every other word in the sequence. It is computed using three matrices: Query (Q), Key (K), and Value (V), which are computed from the input. The attention mechanism is defined as:

Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V

This allows the model to weigh the importance of different words in the sequence and focus on the most relevant ones when generating the output.


In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Hard Negative Mining Technqiues", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Hard Negative Mining Technqiues'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 11.71it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents ✅


Answer: There is no mention of Hard Negative Mining Techniques in the provided context.
Sources: [{'source': 'doc_b3c610dd_1', 'page': 'unknown', 'score': 0.3520892620096737, 'preview': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechanisms alone are sufficient for sequence modelling. No RNN or CNN needed.\nEnables full parallelisation during training....'}, {'source': 'doc_67473df9_1', 'page': 'unknown', 'score': 0.3520892620096737, 'preview': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechanisms alone are sufficient for sequence modelling. No RNN or CNN needed.\nEnables full parallelisation during training....'}, {'source': 'doc_3807162c_1', 'page': 'unknown', 'score': 0.3520892620096737, 'preview': '\x7f BLEU score: 41.0 on WMT 2014 English-to-French\n\x7f Training time: 12 hours on 8 P100 GPUs\nKey Takeaway\nAttention mechani

In [22]:
# Step 1: Imports
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# Step 2: LLM
groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

# Step 3: Embedding Manager
embedding_manager = EmbeddingManager()

# Step 4: Vectorstore
vectorstore = VectorStore()

# Step 5: RAG Retriever
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

print("Sab ready hai! ✅")

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1995.25it/s]


Model loaded successfully. Embedding dimension: 384
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 30
Sab ready hai! ✅


C:\Users\hhh\AppData\Local\Temp\ipykernel_6076\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [26]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("What are embeddings", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What are embeddings'
Top K: 3
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 17.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents ✅
Streaming answer:
Use the following context to answer the question concisely.
Context:
Text Embedd

ings in NLP
What are Embeddings?
Embeddings convert words or sentences into dense numerical vectors. Similar words end up
close together in vector space. They c

apture semantic meaning.
Types of Embeddings
 Word2Vec: predicts surrounding words (CBOW) or centre word (Skip-gram)
 GloVe: Global Vectors — uses word co-occurrence statistics
 FastText: handles subwords, works on unseen/rare words
 BERT Embeddings: contextual — same word gets different vectors in different sentences
Word2Vec Example
 king - man + woman ≈ queen
 paris - france + italy ≈ rome
 Vector dimension: typically 100–300
How to Use (Python)
 from gensim.models import Word2Vec
 model = Word2Vec(sentences, vector_size=100, window=5)
 vector = model.wv['python'] # get embedding for 'python'
Applications
 Semantic search and document similarity
 Sentiment analysis and text classification
 Machine translation and question answering

Text Embeddings in NLP
What are Embeddings?
Embeddings convert words or sentences into dense numerical vectors. Similar words end up
close together in vector space. They capture semantic meaning.
Types of Embeddings
 Word2Vec: predicts surr